# 用 Python 从零构建 RAG 系统（审查修订版）

本 Notebook 把「检索增强生成」的完整链路拆成 7 个阶段，每个阶段是一个带中文注释
的独立函数；所有可调参数集中在第 1 个代码格的 `Config` 中。

| 阶段 | 做什么 | 代码格 |
| --- | --- | --- |
| 0 | 配置与依赖初始化（模型 / 网关只加载一次） | 第 1 格 |
| 1 | 读取文档并按空行切分成块（Chunk） | 第 2 格 |
| 2 | 用向量模型把文本转成向量（Embedding） | 第 3 格 |
| 3 | 把「块 + 向量」写入向量数据库 Chroma | 第 4 格 |
| 4 | 根据问题检索最相似的块（Retrieve） | 第 5 格 |
| 5 | 用 CrossEncoder 对候选块重排序（Rerank） | 第 6 格 |
| 6 | 把问题 + 精选片段交给大模型生成答案（Generate） | 第 7 格 |
| 7 | 串起整条流水线并执行一次 | 第 8 格 |

**相对原版的改动（对应审查编号）**：修复循环里打印整个列表的 Bug（R01）、清除
7 万余字符调试输出（R02/R13）、执行计数复位为 null（R03）、配置与密钥集中管理并
支持 `.env`（R04）、补齐依赖声明（R05）、消除跨格全局变量串联（R06/R10）、模型只
加载一次（R07）、空结果与空库防御（R08）、改用 `upsert` 保证重复运行不冲突（R09）、
批量编码（R12）、网关调用加异常处理（R11）、删除空单元格（R14）、统一 PEP8 格式
与引号/注释（R15–R21）、补充 Markdown 说明（R22）、忽略 checkpoint 与备份（R23）。

> **运行前提**：本地网关 `http://127.0.0.1:7863/v1` 已启动；两个 HuggingFace 模型
> 已提前下载到缓存。换成真实云 API 只需改 `Config` 的三个字段（或写进 `.env`）。
>
> **运行方式**：直接 `Run All`（整条链路只跑一次）。想单步查看某个阶段，在任意新
> 单元格里调用对应函数即可，例如 `load_chunks()[:2]` 或 `retrieve("你的问题", collection)`
> ——最后一格返回的 `results` 字典里也保存了全部中间结果。

## 第 1 格 · 配置与依赖

把所有「魔法值」收拢到一个 `@dataclass` 里：改路径 / 模型 / 网关地址 / Top-K
只动这一处（R04/R20）。`.env` 由 `python-dotenv` 读取，密钥不再硬编码（R04）；
`load_dotenv()` 默认**不覆盖**已存在的环境变量，方便命令行临时覆盖。

模型与客户端是「只读共享资源」，在模块级只实例化一次（R07）。Notebook 里这种
模块级初始化是常规做法；若要把本文件当库 import，可把初始化段包进
`if __name__ == "__main__":` 并额外提供一个 `init_resources()` 函数。

In [ ]:
# ── 标准库 ──────────────────────────────────────────────────────────────
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Sequence, Tuple

# ── 第三方库（版本见 pyproject.toml，对应审查 R05）───────────────────────
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import CrossEncoder, SentenceTransformer

# 读取项目根目录下的 .env；文件不存在时静默跳过，不会报错。
load_dotenv()


@dataclass(frozen=True)
class Config:
    """RAG 流水线的全部可调参数，集中管理避免散落各处（R04/R20/R21）。"""

    # 文档与向量库
    doc_path: Path = Path("doc.md")
    collection_name: str = "rag_demo"

    # 两个模型：首次运行自动下载，之后走本地缓存。
    embedding_model_name: str = "shibing624/text2vec-base-chinese"
    rerank_model_name: str = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

    # 本地网关（OpenAI 兼容）。api_key 在本地网关里只是占位符，但 openai 库
    # 要求非空字符串；真实云 API 请把密钥写进 .env 的 RAG_API_KEY。
    api_key: str = field(default_factory=lambda: os.getenv("RAG_API_KEY", "test_key"))
    base_url: str = field(default_factory=lambda: os.getenv("RAG_BASE_URL", "http://127.0.0.1:7863/v1"))
    llm_model: str = field(default_factory=lambda: os.getenv("RAG_MODEL", "global:deepseek-v4.1-flash"))

    # 检索 / 重排 / 生成的规模参数
    retrieve_top_k: int = 5        # 向量库初筛返回的块数
    rerank_top_k: int = 3          # 重排后真正喂给大模型的块数
    gateway_timeout: float = 60.0  # 网关超时（秒），避免网络异常时无限等待


CFG = Config()

# 模型是「重资源」：只实例化一次，后续函数反复复用（R07）。
print("[init] 正在加载向量模型（首次运行需下载）…")
EMBED_MODEL = SentenceTransformer(CFG.embedding_model_name)
print("[init] 正在加载重排模型（首次运行需下载）…")
RERANK_MODEL = CrossEncoder(CFG.rerank_model_name)

# OpenAI 客户端同样只建一次：它内部维护连接池（R06）。
LLM_CLIENT = OpenAI(api_key=CFG.api_key, base_url=CFG.base_url, timeout=CFG.gateway_timeout)
print(f"[init] 完成 | 模型={CFG.llm_model} | 网关={CFG.base_url}")

## 第 2 格 · 阶段 1：加载文档并切块

`doc.md` 用两个连续换行 `\n\n` 分段。这里显式指定 `encoding="utf-8"`（Windows
默认是 GBK，会乱码），并去掉每段首尾空白、过滤空段，避免把空字符串塞进向量库。
原版在循环里打印了整个 `chunks` 列表（R01），修订后只打印「块数 + 首块长度」。

In [ ]:
def load_chunks(doc_path: Path = CFG.doc_path) -> List[str]:
    """读取 Markdown 文档，按空行切分成文本块并返回列表。

    参数
    ----
    doc_path : Path
        文档路径，默认取 Config.doc_path。
    返回
    ----
    List[str]
        非空文本块，顺序与原文一致。
    """
    content = doc_path.read_text(encoding="utf-8")  # 显式编码，避免 Windows 默认 GBK 乱码
    raw_chunks = content.split("\n\n")            # 两个连续换行 = 一个段落分隔符
    # 列表推导：strip 去首尾空白，if 过滤空段（R12：去掉多余的中间变量）
    chunks = [c.strip() for c in raw_chunks if c.strip()]
    print(f"[1/7] 文档切块完成：共 {len(chunks)} 块，首块 {len(chunks[0])} 字符")
    return chunks

## 第 3 格 · 阶段 2：文本向量化（Embedding）

把每个文本块映射成一串浮点数（向量）：语义相近的文本，向量距离也近。
`encode()` 支持一次传列表**批量**编码，比逐条调用快得多（R12）；
`convert_to_numpy=True` 后统一 `.tolist()` 成普通 Python 列表，方便写入 Chroma。
问题与文档必须用**同一个**模型编码，距离才有意义。

In [ ]:
def embed_texts(texts: Sequence[str]) -> List[List[float]]:
    """把一批文本编码为向量列表。

    参数
    ----
    texts : Sequence[str]
        待编码的文本（chunk 或 query）。
    返回
    ----
    List[List[float]]
        与输入一一对应的向量，维度由模型决定（此处 768 维）。
    """
    # batch_size=32 控制显存/内存占用；show_progress_bar=False 保持输出干净（R13）。
    vectors = EMBED_MODEL.encode(
        list(texts),
        batch_size=32,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    return vectors.tolist()


def embed_one(text: str) -> List[float]:
    """单条文本的便捷封装：复用批量函数，避免重复实现（R19 去重）。"""
    return embed_texts([text])[0]

## 第 4 格 · 阶段 3：写入向量数据库

Chroma 的 `EphemeralClient()` 把数据放在内存里，进程结束即清空——对教学示例刚好
够用，且每次从头运行都是干净状态。换成 `PersistentClient(path=...)` 即可落盘。

关键改动（R09/R10）：
- 用 `upsert` 代替 `add`：id 已存在就覆盖，重复运行不会因主键冲突报错；
- collection 由函数**返回**、后续函数**显式接收**，不再偷偷依赖外部全局变量；
- 校验「块数 == 向量数」，数据错位时立刻报错而不是静默写坏。

In [ ]:
def build_collection(chunks: List[str], embeddings: List[List[float]], name: str = CFG.collection_name):
    """创建内存版 Chroma collection 并写入全部「块 + 向量」。

    返回
    ----
    chromadb.api.models.Collection.Collection
        可直接用于 query() 的集合对象。
    """
    if len(chunks) != len(embeddings):
        raise ValueError(f"块数({len(chunks)}) 与向量数({len(embeddings)}) 不一致")

    client = chromadb.EphemeralClient()
    collection = client.get_or_create_collection(name=name)

    ids = [str(i) for i in range(len(chunks))]  # 稳定 id：0,1,2… 与块下标对应
    collection.upsert(ids=ids, documents=chunks, embeddings=embeddings)
    print(f"[3/7] 入库完成：collection='{name}'，count={collection.count()}")
    return collection

## 第 5 格 · 阶段 4：向量检索（Retrieve）

把问题编码成向量，让 Chroma 返回距离最近的 `top_k` 个块。

**关于审查里的 R08（已在复核时部分驳回）**：实测 chromadb 1.5.9 的 `query()`
默认 `include` 就包含 `documents`，所以**会**返回文本，无需手动补 `include`。
这里仍保留空结果防御——库为空 / 无命中时返回 `[]`，不会让
`results["documents"][0]` 抛 IndexError。

In [ ]:
def retrieve(query: str, collection, top_k: int = CFG.retrieve_top_k) -> List[str]:
    """在向量库中检索与 query 最相似的 top_k 个文本块。

    参数
    ----
    query : str
        用户问题。
    collection : Collection
        build_collection() 返回的集合。
    top_k : int
        初筛返回数量。
    """
    if collection.count() == 0:
        print("[4/7] 向量库为空，跳过检索")
        return []

    query_vector = embed_one(query)            # 问题与文档用同一模型编码，才可比
    n = min(top_k, collection.count())        # 防止请求条数超过库中总量
    results = collection.query(query_embeddings=[query_vector], n_results=n)

    # query() 默认已含 documents；仍做 None / 空列表防御，保证任何版本都不崩（R08）。
    docs = results.get("documents") or []
    hits = docs[0] if docs else []
    print(f"[4/7] 检索完成：返回 {len(hits)} 个候选块")
    return list(hits)

## 第 6 格 · 阶段 5：重排序（Rerank）

向量检索是「双塔模型」，速度快但精度有限；CrossEncoder 会把 `(问题, 块)` 拼在
一起过一次 Transformer，打分更准，适合对少量候选做二次排序。

模型已在第 1 格加载（R07），这里只做打分 + 排序：`zip` 把分数配回原文本，
用 `sorted` 按分数降序取前 `top_k`（不改动原列表，比原地 `sort` 更安全）。
空候选时直接返回空列表。

In [ ]:
def rerank(query: str, candidates: List[str], top_k: int = CFG.rerank_top_k) -> List[str]:
    """用 CrossEncoder 对候选块重新打分，返回最相关的 top_k 个。

    参数
    ----
    query : str
        用户问题。
    candidates : List[str]
        retrieve() 初筛出的候选块。
    top_k : int
        重排后保留的块数（真正进入 Prompt 的数量）。
    """
    if not candidates:
        print("[5/7] 无候选块，跳过重排")
        return []

    pairs: List[Tuple[str, str]] = [(query, c) for c in candidates]
    scores = RERANK_MODEL.predict(pairs, show_progress_bar=False)

    ranked = sorted(zip(candidates, scores), key=lambda item: item[1], reverse=True)
    kept = [chunk for chunk, _ in ranked[:top_k]]
    print(f"[5/7] 重排完成：{len(candidates)} → {len(kept)} 块")
    return kept

## 第 7 格 · 阶段 6：生成答案（Generate）

把「问题 + 重排后的片段」拼成 Prompt，交给 OpenAI 兼容网关生成回答。

改动要点：
- Prompt 用括号拼接而非整段三引号，不再有行尾空格 / 缩进污染（R17）；
- 去掉调试用的整段 Prompt 打印（R13），只打印调用状态；
- `try/except` 包住网络调用（R11）：网关没开 / 超时 / 模型名错误时，返回一段
  可读说明而不是抛堆栈让 Notebook 中断；
- `response.choices[0].message.content` 可能为 None，用 `or ""` 兜底。

In [ ]:
def generate(query: str, chunks: List[str]) -> str:
    """基于检索到的片段，调用大模型生成答案。

    参数
    ----
    query : str
        用户问题。
    chunks : List[str]
        rerank() 精选出的片段，作为「事实依据」注入 Prompt。
    返回
    ----
    str
        模型答案；调用失败时返回以 [网关调用失败] 开头的说明文本。
    """
    if not chunks:
        return "（未检索到任何相关片段，无法生成答案）"

    context = "\n\n".join(f"- {c}" for c in chunks)  # 每个片段加项目符号，便于模型区分
    prompt = (
        "你是一位知识助手，请根据用户的问题和下列片段生成准确的回答。\n\n"
        f"用户问题：{query}\n\n"
        f"相关片段：\n{context}\n\n"
        "请基于上述内容作答，不要编造信息。"
    )

    try:
        response = LLM_CLIENT.chat.completions.create(
            model=CFG.llm_model,
            messages=[{"role": "user", "content": prompt}],
            stream=False,
        )
        answer = (response.choices[0].message.content or "").strip()
        print(f"[6/7] 生成完成：{len(answer)} 字符")
        return answer
    except Exception as exc:  # 网络/认证/模型名错误都在此收敛，保证流水线不中断（R11）
        msg = f"[网关调用失败] {type(exc).__name__}: {exc}"
        print(msg)
        return msg

## 第 8 格 · 阶段 7：串起整条流水线（只执行一次）

`main()` 按 1→7 顺序显式调用上面的函数，是唯一「从头到尾执行」的入口，
每一步都把结果作为参数传给下一步（R06：无隐式全局串联）。

它返回一个 `results` 字典，保存**全部中间结果**，方便你在后续单元格里检查：
`results["chunks"]` / `results["retrieved"]` / `results["reranked"]` / `results["answer"]`。
上面 6 个阶段格只定义函数、不单独执行，因此 `Run All` 整条链路恰好跑一次，
不会有重复的模型调用与重复输出。

想换问题：改 `main("你的问题")`；想换模型 / 地址 / Top-K：改第 1 格的 `Config`。

In [ ]:
def main(query: str = "落地成效实现了哪些价值？") -> dict:
    """执行完整 RAG 流水线，返回含全部中间结果的字典。

    参数
    ----
    query : str
        用户问题。
    返回
    ----
    dict
        键：query / chunks / embeddings / collection / retrieved / reranked / answer
    """
    print("=" * 60)
    print("RAG 流水线开始")
    print("=" * 60)

    chunks = load_chunks()                             # 1. 切块
    embeddings = embed_texts(chunks)                  # 2. 向量化
    collection = build_collection(chunks, embeddings)  # 3. 入库
    retrieved = retrieve(query, collection)           # 4. 检索
    reranked = rerank(query, retrieved)               # 5. 重排
    answer = generate(query, reranked)                # 6. 生成

    print("=" * 60)
    print(f"[7/7] 完成 | 问题：{query}")
    print("=" * 60)
    return {
        "query": query,
        "chunks": chunks,
        "embeddings": embeddings,
        "collection": collection,
        "retrieved": retrieved,
        "reranked": reranked,
        "answer": answer,
    }


if __name__ == "__main__":
    # 在 Jupyter 中 __name__ 同样是 "__main__"，所以 Run All 会执行这里；
    # 把本文件当模块 import 时则不会触发，保持无副作用。
    results = main()
    print("\n===== 最终答案 =====")
    print(results["answer"])
    print("\n提示：results 字典保存了全部中间结果，"
          "例如在新单元格执行 results['reranked'] 可查看重排后的片段。")